<!-- CONCLUSION-CELL -->
> ## ⏳ 결론 — 아직 실행 전 (Colab에서 Run All 후 이 셀을 채워주세요)
>
> 이 노트북은 "릴리스 1프레임 → 투구 전체 프레임 시퀀스" 아이디어를 검증하려고 새로
> 작성됐습니다. 로컬 환경엔 실제 영상·좌표·GPU가 없어 실행 결과를 미리 알 수 없으므로,
> 다른 노트북과 달리 이 CONCLUSION-CELL엔 가짜 숫자를 채워넣지 않았습니다.
>
> **Run All 후 아래를 채워 넣어주세요** (맨 아래 "7. 결과 저장" 셀이 출력하는 summary 참고):
>
> | 항목 | 값 |
> |---|---|
> | E11-1 정형 단독 Val R² | ? |
> | E11-2 영상 시퀀스 단독 Val R² | ? |
> | E11-3 정형+영상임베딩 융합 Val R² | ? |
> | E11-4 30-seed paired t-test | 평균차 ?, t=?, p=? |
> | 결론 (채택 / 기각) | ? |
>
> ⚠ **참고**: 훨씬 단순한 구조(15번, Statcast 5피처×15투구 1D-CNN)도 더 많은 학습
> 데이터로 XGB에 30/0으로 완패했고(`sequence_model_results.json`), 정적 릴리스-1프레임
> 영상 피처 융합(12번)도 유의하게 악화(p<0.0001)됐습니다. bio-매칭 경기 수(~1,300~3,800개)는
> 정형 단독 학습 데이터(13,871개)보다 훨씬 적어 이번 2단 구조는 과적합 위험이 더 큽니다.
> **기각되더라도** 그 자체가 유효한 결과이니 정직하게 기록해주세요 — "정지 1프레임을
> 시퀀스로 바꿔도 안 됐다"는 것도 이 프로젝트가 반복해온 검증 방식의 연장선입니다.


# 17. 영상 시퀀스 모델 실험 (Phase 12 — 로드맵 E, 릴리스 1프레임 → 전체 프레임 시퀀스)

**동기**: 12번(정적 릴리스-1프레임 → 9-stat 집계 융합)은 유의하게 기각됐고
(`bio_experiment_results.csv`: 정형 0.0659 vs 융합 0.0566, p<0.0001), README도 원인으로
"정지 1프레임이라 폼의 변화·타이밍을 못 잡는다"를 지목하며 다음 단계로
"릴리스 전후 시퀀스(동작 궤적)"를 제안했습니다. 이 노트북은 그 제안을 실제로 구현합니다.

**입력**: `05_video_sequence_pipeline.ipynb`가 만든 경기 단위 텐서
(`video_seq_pitch{N}_T{T}.npz`) — (경기, 최대15투구, T=32프레임, 각도9종).

**⚠ 투구 축은 순서가 없다**: `video_sequence_features.py` 상단 docstring 참고 — 02번
크롤링이 실제 pitch_number를 보존하지 않아, 경기 내 투구 순서를 신뢰할 수 없습니다.
그래서 이 노트북의 모델은 프레임 축(투구 1개 내부, 진짜 시간순)엔 1D-CNN을 쓰지만,
투구 축(경기 내 여러 투구)엔 순서를 학습하는 레이어 대신 **masked mean/std pooling**
(순서 무관, permutation-invariant)을 씁니다.

| 실험 | 내용 |
|---|---|
| E11-1 | 정형 단독 XGB (13번 채택 모델과 동일 조건, 이 병합 subset 기준 참고용 재확인) |
| E11-2 | 영상 시퀀스 모델 단독 (PitchFrameCNN + masked pooling, 단판) |
| E11-3 | 영상 임베딩 추출 → 정형과 융합 (12번의 손수집계 81피처 대신 학습된 임베딩 사용) |
| 🔬 E11-4 | 정형 단독 vs 정형+영상임베딩 융합 **paired t-test** (n=30 seeds) |
| E11-5 | SHAP — 영상 임베딩 feature 순위 확인 |

> **리스크 고지**: bio-매칭 경기 수(~1,300~3,800개)는 정형 학습 데이터(13,871개)보다
> 훨씬 적습니다. 15번(더 단순한 1단 CNN)도 더 많은 데이터로 XGB에 완패했습니다.
> 이 실험이 기각되더라도(오히려 그럴 가능성이 높습니다) 그 자체가 유효한 결과입니다.

**선행 조건**: `05_video_sequence_pipeline.ipynb` 완료(`video_seq_pitch15_T32.npz` 존재),
`10_tuning_experiment.ipynb`의 `best_params.json`(없으면 기본 파라미터로 대체).


In [ ]:
import os, sys

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    DRIVE    = '/content/drive/MyDrive/MLB_pitcher'
    DATA     = os.path.join(DRIVE, 'data')
    STAT_DIR = next((d for d in [os.path.join(DATA, 'features'), os.path.join(DATA, '4_features')]
                     if os.path.exists(os.path.join(d, 'features_pitch15.parquet'))),
                    os.path.join(DATA, 'features'))
    SEQ_DIR    = os.path.join(DATA, '4_features')   # 05_video_sequence_pipeline.ipynb 출력 위치
    OUTPUT_DIR = os.path.join(DRIVE, '4_output')
else:
    DRIVE      = os.path.abspath('.')  # 로컬 실행 시 데이터 루트로 변경
    STAT_DIR   = os.path.join(DRIVE, '0_data', '4_features')
    SEQ_DIR    = os.path.join(DRIVE, '0_data', '4_features')
    OUTPUT_DIR = os.path.join(DRIVE, '4_output')

STATCAST_PATH = os.path.join(STAT_DIR, 'features_pitch15.parquet')
SEQ_MAX_PITCHES, SEQ_T = 15, 32   # video_sequence_features.MAX_PITCHES / T_RESAMPLE 기본값과 일치해야 함
SEQ_NPZ_PATH  = os.path.join(SEQ_DIR, f'video_seq_pitch{SEQ_MAX_PITCHES}_T{SEQ_T}.npz')
SEQ_META_PATH = os.path.join(SEQ_DIR, f'video_seq_pitch{SEQ_MAX_PITCHES}_T{SEQ_T}_meta.parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'환경: {"코랩" if IN_COLAB else "로컬"}')
print(f'Statcast : {STATCAST_PATH}  (존재: {os.path.exists(STATCAST_PATH)})')
print(f'Seq npz  : {SEQ_NPZ_PATH}  (존재: {os.path.exists(SEQ_NPZ_PATH)})')
print(f'Seq meta : {SEQ_META_PATH}  (존재: {os.path.exists(SEQ_META_PATH)})')

if not os.path.exists(STATCAST_PATH):
    raise FileNotFoundError(f'정형 피처 파일 없음: {STATCAST_PATH}')
if not os.path.exists(SEQ_NPZ_PATH):
    raise FileNotFoundError(f'영상 시퀀스 텐서 없음: {SEQ_NPZ_PATH} — 05_video_sequence_pipeline.ipynb 먼저 실행')


In [ ]:
try:
    import xgboost, shap, torch
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'shap', 'torch', '-q'])

import json
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')
if DEVICE == 'cpu':
    print('  ⚠ GPU 미사용. [런타임 → 런타임 유형 변경 → T4 GPU] 후 재실행 권장(경기 텐서가 커서 CPU는 느림)')

# 10_tuning_experiment 결과의 최적 파라미터 로드 — 없으면 기본값
BP_PATH = os.path.join(OUTPUT_DIR, 'best_params.json')
if os.path.exists(BP_PATH):
    XGB_PARAMS = json.load(open(BP_PATH, encoding='utf-8'))['XGB']
    print('best_params.json 로드:', XGB_PARAMS)
else:
    XGB_PARAMS = dict(n_estimators=300, learning_rate=0.05, max_depth=6,
                      subsample=0.8, colsample_bytree=0.8)
    print('best_params.json 없음 → 기본 파라미터 사용')

XGB_FIXED = dict(random_state=42, n_jobs=-1, verbosity=0, early_stopping_rounds=50)
print('패키지/파라미터 로드 완료')


## 1. 데이터 로드 및 병합

In [ ]:
META_COLS = ['game_pk', 'pitcher', 'season', 'y_whiff']

df_stat = pd.read_parquet(STATCAST_PATH)
STAT_COLS = [c for c in df_stat.columns if c not in META_COLS]

npz = np.load(SEQ_NPZ_PATH)
X_all, X_rel_all, mask_all = npz['X'], npz['X_rel'], npz['mask']
meta = pd.read_parquet(SEQ_META_PATH)
print(f'Statcast : {df_stat.shape}  columns={len(df_stat.columns)}')
print(f'Video seq: X{X_all.shape}  meta{meta.shape}')

# 정형 피처와 영상 텐서를 (game_pk, pitcher, season) 기준으로 이너 조인
# → 두 데이터에 모두 존재하는 경기만 사용(정형은 23,225경기, 영상은 그중 매칭분만)
merge_key = df_stat[META_COLS].reset_index().rename(columns={'index': 'stat_row'})
meta_idx  = meta.reset_index().rename(columns={'index': 'seq_row'})
joined = merge_key.merge(meta_idx, on=['game_pk', 'pitcher', 'season'], how='inner')
print(f'\n병합: 정형 {len(df_stat)}행 중 영상매칭 {len(joined)}행')

stat_rows = joined['stat_row'].to_numpy()
seq_rows  = joined['seq_row'].to_numpy()

X     = X_all[seq_rows]       # (N, 15, 32, 9)
X_rel = X_rel_all[seq_rows]   # (N, 15)  -- 이번 실험에선 미사용(향후 확장용으로 같이 로드만 해둠)
mask  = mask_all[seq_rows]    # (N, 15)
df    = df_stat.iloc[stat_rows].reset_index(drop=True)
n_pitches_used = joined['n_pitches_used'].to_numpy()

print(f'경기당 평균 매칭 투구 수: {n_pitches_used.mean():.1f} / {SEQ_MAX_PITCHES}')

season  = df['season'].to_numpy()
tr_mask = np.isin(season, [2021, 2022, 2023])
vl_mask = season == 2024
te_mask = season == 2025
print(f'train {tr_mask.sum():,} | val {vl_mask.sum():,} | test {te_mask.sum():,}')


## 2. 모델 정의 (PitchFrameCNN + masked pooling)

In [ ]:
class PitchFrameCNN(nn.Module):
    """투구 1개의 프레임 시퀀스 (T, 9각도) → 고정 크기 임베딩.
    15_sequence_model_experiment.ipynb의 PitchCNN과 동일 구조(Conv1d×2 + AdaptiveAvgPool1d + Linear).
    입력 채널만 Statcast 5피처 → 영상 각도 9종으로 바뀌었다.
    """
    def __init__(self, n_feat=9, embed_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_feat, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64, embed_dim), nn.ReLU(),
        )

    def forward(self, x):        # x: (B, T, F)
        x = x.transpose(1, 2)     # → (B, F, T)
        return self.head(self.net(x))  # (B, embed_dim)


class GameVideoModel(nn.Module):
    """경기 단위 예측: 투구별 프레임-CNN 임베딩 → masked mean/std pooling(순서 무관) → 회귀.

    ⚠ 투구 축은 순서가 없다(video_sequence_features.py 참고) → CNN/LSTM으로 투구 순서를
    학습하게 하면 존재하지 않는 신호를 학습하는 셈이라, 여기선 순서-불변 pooling만 쓴다.
    """
    def __init__(self, n_feat=9, embed_dim=16):
        super().__init__()
        self.pitch_encoder = PitchFrameCNN(n_feat=n_feat, embed_dim=embed_dim)
        self.embed_dim = embed_dim
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim * 2 + 1, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def pooled_embedding(self, X, mask):
        """(mean_emb, std_emb, n_valid) 풀링 결과 반환 — 회귀뿐 아니라 XGB 융합용
        고정길이 feature 추출에도 재사용한다(E11-3)."""
        B, P, T, F = X.shape
        flat = X.reshape(B * P, T, F)
        emb = self.pitch_encoder(flat).reshape(B, P, self.embed_dim)  # (B, P, E)

        m = mask.unsqueeze(-1)                                  # (B, P, 1)
        n_valid = mask.sum(dim=1, keepdim=True).clamp(min=1)   # (B, 1)

        mean_emb = (emb * m).sum(dim=1) / n_valid
        var_emb  = ((emb - mean_emb.unsqueeze(1)) ** 2 * m).sum(dim=1) / n_valid
        std_emb  = torch.sqrt(var_emb + 1e-6)
        return torch.cat([mean_emb, std_emb, n_valid], dim=-1)  # (B, 2E+1)

    def forward(self, X, mask):
        return self.regressor(self.pooled_embedding(X, mask)).squeeze(-1)


def train_video_model(Xtr, masktr, ytr, Xvl, maskvl, yvl,
                       seed=42, embed_dim=16, epochs=60, patience=8, bs=64, lr=1e-3, verbose=False):
    """early stopping 학습 → best val 기준 (val_r2, model) 반환. 15번의 train_cnn과 동일 패턴."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = GameVideoModel(n_feat=Xtr.shape[-1], embed_dim=embed_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr, device=DEVICE); masktr_t = torch.tensor(masktr, device=DEVICE)
    ytr_t = torch.tensor(ytr, device=DEVICE)
    Xvl_t = torch.tensor(Xvl, device=DEVICE); maskvl_t = torch.tensor(maskvl, device=DEVICE)
    n = len(Xtr_t)

    best_r2, best_state, wait = -1e9, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = lossf(model(Xtr_t[idx], masktr_t[idx]), ytr_t[idx])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vpred = model(Xvl_t, maskvl_t).cpu().numpy()
        vr2 = r2_score(yvl, vpred)
        if vr2 > best_r2:
            best_r2, best_state, wait = vr2, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
        if verbose and ep % 10 == 0:
            print(f'  ep{ep:3d}  val_R²={vr2:.4f}')
    model.load_state_dict(best_state)
    return best_r2, model

print('모델 정의 완료: PitchFrameCNN, GameVideoModel, train_video_model')


## 3. E11-1 / E11-2: 정형 단독 vs 영상 시퀀스 단독 (단판)

In [ ]:
y_arr = df['y_whiff'].to_numpy().astype('float32')
X_f32 = X.astype('float32')

X_tr, mask_tr, y_tr = X_f32[tr_mask], mask[tr_mask], y_arr[tr_mask]
X_vl, mask_vl, y_vl = X_f32[vl_mask], mask[vl_mask], y_arr[vl_mask]
X_te, mask_te, y_te = X_f32[te_mask], mask[te_mask], y_arr[te_mask]

# E11-1: 정형 단독 XGB (13번 채택 모델과 동일 조건 — 이 병합 subset 기준으로 재확인)
m_stat = xgb.XGBRegressor(**{**XGB_PARAMS, **XGB_FIXED, 'random_state': 42})
m_stat.fit(df.loc[tr_mask, STAT_COLS], y_tr, eval_set=[(df.loc[vl_mask, STAT_COLS], y_vl)], verbose=False)
stat_vr2 = r2_score(y_vl, m_stat.predict(df.loc[vl_mask, STAT_COLS]))
stat_tr2 = r2_score(y_te, m_stat.predict(df.loc[te_mask, STAT_COLS]))
print(f'[E11-1 정형 단독]        Val R²={stat_vr2:.4f}  Test R²={stat_tr2:.4f}  (병합 subset={len(df):,}경기 기준)')

# E11-2: 영상 시퀀스 단독 (단판 — 15번처럼 무겁고, E11-3 임베딩 추출용 학습을 겸함)
video_vr2, video_model = train_video_model(X_tr, mask_tr, y_tr, X_vl, mask_vl, y_vl, seed=42, verbose=True)
with torch.no_grad():
    video_te_pred = video_model(torch.tensor(X_te, device=DEVICE),
                                 torch.tensor(mask_te, device=DEVICE)).cpu().numpy()
video_tr2 = r2_score(y_te, video_te_pred)
print(f'\n[E11-2 영상 시퀀스 단독]  Val R²={video_vr2:.4f}  Test R²={video_tr2:.4f}')


## 4. E11-3: 영상 임베딩 추출 → 정형과 융합

In [ ]:
# E11-2에서 학습한 video_model(seed=42)로 전체 경기의 pooled embedding 추출.
# 12번(손수집계 81피처)과 달리, 여기선 딥러닝이 학습한 임베딩을 "고정 feature"로 취급해 XGB에 넣는다
# (임베딩 자체는 seed=42 1회 학습분을 재사용 — 12번의 hand-crafted feature와 동일하게 취급).
def extract_embeddings(model, X_arr, mask_arr, bs=512):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X_arr), bs):
            xb = torch.tensor(X_arr[i:i + bs], device=DEVICE)
            mb = torch.tensor(mask_arr[i:i + bs], device=DEVICE)
            outs.append(model.pooled_embedding(xb, mb).cpu().numpy())
    return np.concatenate(outs, axis=0)

emb_all = extract_embeddings(video_model, X_f32, mask)
EMBED_COLS = ([f'vidseq_emb_mean_{i}' for i in range(video_model.embed_dim)]
              + [f'vidseq_emb_std_{i}' for i in range(video_model.embed_dim)]
              + ['vidseq_n_pitches'])
emb_df = pd.DataFrame(emb_all, columns=EMBED_COLS)
df_fused = pd.concat([df.reset_index(drop=True), emb_df], axis=1)
ALL_COLS = STAT_COLS + EMBED_COLS
print(f'임베딩 컬럼 {len(EMBED_COLS)}개 추가 → 융합 feature 총 {len(ALL_COLS)}개')

m_fused = xgb.XGBRegressor(**{**XGB_PARAMS, **XGB_FIXED, 'random_state': 42})
m_fused.fit(df_fused.loc[tr_mask, ALL_COLS], y_tr, eval_set=[(df_fused.loc[vl_mask, ALL_COLS], y_vl)], verbose=False)
fused_vr2 = r2_score(y_vl, m_fused.predict(df_fused.loc[vl_mask, ALL_COLS]))
fused_tr2 = r2_score(y_te, m_fused.predict(df_fused.loc[te_mask, ALL_COLS]))
print(f'[E11-3 정형+영상임베딩 융합]  Val R²={fused_vr2:.4f}  Test R²={fused_tr2:.4f}')

print('\n=== 단판 비교 요약 ===')
print(f'  E11-1 정형 단독         Val R²={stat_vr2:.4f}  Test R²={stat_tr2:.4f}')
print(f'  E11-2 영상 시퀀스 단독   Val R²={video_vr2:.4f}  Test R²={video_tr2:.4f}')
print(f'  E11-3 정형+영상임베딩    Val R²={fused_vr2:.4f}  Test R²={fused_tr2:.4f}')


## 5. 🔬 E11-4 Paired t-test — 정형 vs 정형+영상임베딩 (n=30 seeds)

In [ ]:
# 12번(E7-5)과 동일 방식: 임베딩 자체는 고정(위에서 1회 추출), XGB의 random_state만 30개 시드로 반복.
N_SEEDS = 30
r2_stat_list, r2_fused_list = [], []

for seed in range(N_SEEDS):
    p = {**XGB_PARAMS, **XGB_FIXED, 'random_state': seed}

    ms = xgb.XGBRegressor(**p)
    ms.fit(df.loc[tr_mask, STAT_COLS], y_tr, eval_set=[(df.loc[vl_mask, STAT_COLS], y_vl)], verbose=False)
    r2_stat_list.append(r2_score(y_vl, ms.predict(df.loc[vl_mask, STAT_COLS])))

    mf = xgb.XGBRegressor(**p)
    mf.fit(df_fused.loc[tr_mask, ALL_COLS], y_tr, eval_set=[(df_fused.loc[vl_mask, ALL_COLS], y_vl)], verbose=False)
    r2_fused_list.append(r2_score(y_vl, mf.predict(df_fused.loc[vl_mask, ALL_COLS])))

    if (seed + 1) % 10 == 0:
        print(f'seed {seed+1}/{N_SEEDS} 완료  '
              f'정형={np.mean(r2_stat_list):.4f}  융합={np.mean(r2_fused_list):.4f}')

r2_stat_arr  = np.array(r2_stat_list)
r2_fused_arr = np.array(r2_fused_list)
diff = r2_fused_arr - r2_stat_arr
t_stat, p_value = stats.ttest_rel(r2_fused_arr, r2_stat_arr)

print('\n' + '=' * 52)
print('Paired t-test: 정형 vs 정형+영상시퀀스임베딩')
print('=' * 52)
print(f'정형만    평균 R²: {r2_stat_arr.mean():.4f} ± {r2_stat_arr.std():.4f}')
print(f'정형+영상 평균 R²: {r2_fused_arr.mean():.4f} ± {r2_fused_arr.std():.4f}')
print(f'평균 차이 (융합-정형): {diff.mean():+.4f}')
print(f't-statistic: {t_stat:.4f}  |  p-value: {p_value:.4f}')
if p_value < 0.05:
    verdict = '유의하게 개선 ✅' if diff.mean() > 0 else '유의하게 악화 ❌'
    print(f'→ p < 0.05: {verdict}')
else:
    print('→ p ≥ 0.05: 통계적으로 유의미한 차이 없음 (무의)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.boxplot([r2_stat_arr, r2_fused_arr], tick_labels=['정형\n단독', '정형+\n영상임베딩'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Val R²')
ax.set_title(f'R² 분포 비교 (n={N_SEEDS} seeds)\np={p_value:.4f}')
ax.grid(alpha=0.3)
ax = axes[1]
ax.hist(diff, bins=15, color='darkorange', alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(diff.mean(), color='red', linewidth=1.5, label=f'mean={diff.mean():+.4f}')
ax.set_xlabel('R² 차이 (융합 - 정형)'); ax.set_ylabel('빈도')
ax.set_title('영상시퀀스임베딩 기여 차이 분포'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'video_sequence_ttest.png'), dpi=120, bbox_inches='tight')
plt.show()


## 6. SHAP — 영상 임베딩 feature 순위 확인

In [ ]:
explainer = shap.TreeExplainer(m_fused)
shap_vals = explainer(df_fused.loc[vl_mask, ALL_COLS])
shap_mean = pd.Series(np.abs(shap_vals.values).mean(axis=0), index=ALL_COLS).sort_values(ascending=False)
rank = {f: i + 1 for i, f in enumerate(shap_mean.index)}

print(f'전체 {len(ALL_COLS)}개 feature 중 영상임베딩 feature SHAP 순위 (상위 10개만 표시):\n')
for f in sorted(EMBED_COLS, key=lambda x: rank[x])[:10]:
    print(f'  {rank[f]:3d}위  {f:24s}  SHAP={shap_mean[f]:.5f}')

stat_shap  = shap_mean[STAT_COLS].mean()
embed_shap = shap_mean[EMBED_COLS].mean()
print(f'\n정형 feature 평균 SHAP  : {stat_shap:.5f}')
print(f'영상임베딩 평균 SHAP    : {embed_shap:.5f}')

top = shap_mean.head(20)[::-1]
colors = ['darkorange' if f in EMBED_COLS else 'steelblue' for f in top.index]
plt.figure(figsize=(9, 8))
plt.barh(top.index, top.values, color=colors, alpha=0.85)
plt.xlabel('mean |SHAP|')
plt.title('SHAP Top 20 (주황=영상시퀀스임베딩)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'video_sequence_shap.png'), dpi=120, bbox_inches='tight')
plt.show()


## 7. 결과 저장

In [ ]:
pd.DataFrame({'seed': range(N_SEEDS), 'r2_stat': r2_stat_arr,
              'r2_fused': r2_fused_arr, 'diff': diff}).to_csv(
    os.path.join(OUTPUT_DIR, 'video_sequence_ttest_seeds.csv'), index=False, encoding='utf-8-sig')

summary = {
    'E11-1_stat_only_val_r2':   round(float(stat_vr2), 4),
    'E11-1_stat_only_test_r2':  round(float(stat_tr2), 4),
    'E11-2_video_seq_val_r2':   round(float(video_vr2), 4),
    'E11-2_video_seq_test_r2':  round(float(video_tr2), 4),
    'E11-3_fused_val_r2':       round(float(fused_vr2), 4),
    'E11-3_fused_test_r2':      round(float(fused_tr2), 4),
    'E11-4_ttest_mean_stat':    round(float(r2_stat_arr.mean()), 4),
    'E11-4_ttest_mean_fused':   round(float(r2_fused_arr.mean()), 4),
    'E11-4_mean_diff':          round(float(diff.mean()), 4),
    't_stat':                   round(float(t_stat), 4),
    'p_value':                  round(float(p_value), 4),
    'significant':              bool(p_value < 0.05),
    'n_games_merged':           int(len(df)),
    'avg_pitches_per_game':     round(float(n_pitches_used.mean()), 2),
}
json.dump(summary, open(os.path.join(OUTPUT_DIR, 'video_sequence_results.json'),
                        'w', encoding='utf-8'), ensure_ascii=False, indent=2)

print('저장 완료:')
print('  4_output/video_sequence_ttest_seeds.csv')
print('  4_output/video_sequence_results.json')
print('  4_output/video_sequence_ttest.png')
print('  4_output/video_sequence_shap.png')
print('\n요약:', json.dumps(summary, ensure_ascii=False, indent=2))
print('\n⚠ 위 결과를 이 노트북 맨 위 CONCLUSION-CELL에 정리해 넣어주세요(다른 노트북들의 관행).')
